In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

**Joins :**
- DF1.join(DF2, join_expression, jointype)

In [0]:
customer_data = [(1,'manish','patna',"30-05-2022"),
(2,'vikash','kolkata',"12-03-2023"),
(3,'nikita','delhi',"25-06-2023"),
(4,'rahul','ranchi',"24-03-2023"),
(5,'mahesh','jaipur',"22-03-2023"),
(6,'prantosh','kolkata',"18-10-2022"),
(7,'raman','patna',"30-12-2022"),
(8,'prakash','ranchi',"24-02-2023"),
(9,'ragini','kolkata',"03-03-2023"),
(10,'raushan','jaipur',"05-02-2023")]

customer_schema=['customer_id','customer_name','address','date_of_joining']
customer_df = spark.createDataFrame(data = customer_data, schema = customer_schema)

In [0]:
sales_data = [(1,22,10,"01-06-2022"),
(1,27,5,"03-02-2023"),
(2,5,3,"01-06-2023"),
(5,22,1,"22-03-2023"),
(7,22,4,"03-02-2023"),
(9,5,6,"03-03-2023"),
(2,1,12,"15-06-2023"),
(1,56,2,"25-06-2023"),
(5,12,5,"15-04-2023"),
(11,12,76,"12-03-2023")]

sales_schema=['customer_id','product_id','quantity','date_of_purchase']
sales_df = spark.createDataFrame(data = sales_data, schema = sales_schema)

In [0]:
product_data = [(1, 'fanta',20),
(2, 'dew',22),
(5, 'sprite',40),
(7, 'redbull',100),
(12,'mazza',45),
(22,'coke',27),
(25,'limca',21),
(27,'pepsi',14),
(56,'sting',10)]

product_schema=['id','name','price']
product_df = spark.createDataFrame(data = product_data, schema = product_schema)

_inner Join :_

In [0]:
customer_df.join(sales_df,customer_df["customer_id"] == sales_df["customer_id"], "inner")\
.select(sales_df["product_id"]).sort("product_id").show()

_Left Join :_

In [0]:
customer_df.join(sales_df,customer_df["customer_id"] == sales_df["customer_id"],"left").show()

_Right Join :_

In [0]:
sales_df.join(product_df,sales_df["product_id"] == product_df["id"],"right").show()

_Outer(Full Outer Join) :_

In [0]:
customer_df.join(sales_df,customer_df["customer_id"] == sales_df["customer_id"],"outer").show()

_Left Semi Join :_

In [0]:
# Similar to inner join, but data comes ony from the left side table
# Equivalent to the following sub-query: select * from emp where deptid in (select id from dept)
customer_df.join(sales_df,customer_df["customer_id"] == sales_df["customer_id"],"left_semi").show()

_Left Anti Join :_

In [0]:
# Equivalent to the following sub-query: select * from emp where deptid not in (select id from dept)
customer_df.join(sales_df,customer_df["customer_id"] == sales_df["customer_id"],"left_anti").show()

_Cross Join :_

In [0]:
 joined_df = customer_df.crossJoin(sales_df)
 joined_df.show(joined_df.count())

_Broadcast Hash Join :_

- print(spark.conf.get("spark.sql.autoBroadcastJoinThreshold")) -- returns in byte
- spark.conf.set("spark.sql.autoBroadcastJoinThreshold",-1) -- to disable broadcast Join
- spark.conf.set("spark.sql.autoBroadcastJoinThreshold",in-byte)
- spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 52428800) -- 50MB

In [0]:
broadcast_df = customer_df.join(broadcast(sales_df),customer_df["customer_id"] == sales_df["customer_id"],"inner")
broadcast_df.explain()

In [0]:
df1 = customer_df.select(col("customer_id"))
df1.createOrReplaceTempView("customer")
df2 = sales_df.select(col("customer_id"))
df2.createOrReplaceTempView("sales")


In [0]:
%sql
select c.customer_id as customer_info,s.customer_id as sales_info
from customer c
full outer join sales s
on c.customer_id = s.customer_id